## Load data

In [ ]:
import pandas as pd
import numpy as np

#Load clean transaction data and targets (clean data)
df_train = pd.read_csv("data/train_clean.csv", parse_dates=["order_date","pack_date"])
df_valid = pd.read_csv("data/valid_clean.csv", parse_dates=["order_date","pack_date"])

df_train_targets = pd.read_csv("data/train_targets.csv")
df_valid_targets = pd.read_csv("data/valid_targets.csv")

print("Train transactions:",df_train.shape)
print("Valid transactions:",df_valid.shape)

In [4]:
#Reference date: last day of the transaction period
#We use this to calculate recency (how many days since last purchase)
reference_date = pd.Timestamp("2017-12-31")
reference_date

Timestamp('2017-12-31 00:00:00')

## RFM + pruschase behavior features

In [9]:
def compute_rfm_features(df):
    """
    Computes RFM and purchase behavior features, aggregated at customer level.
    Input: transaction-level dataframe (one row per time)
    Output: one row per customer
    """
    df=df.copy()
    # Separate returned and non-returned items
    df_net = df[df["is_returned"]==0]
    
    #---- Recency: days since last purchase ----
    recency = (
        df.groupby("cust_id")["order_date"]
        .max()
        .apply(lambda x: (reference_date -x).days)
        .rename("recency_days")
    )
    #---- Frequency: number of unique orders with at least one non-returned item ---
    #sale_id is the order identifier; one order can have multiple items
    frequency = (
        df_net.groupby("cust_id")["sale_id"]
        .nunique()
        .rename("frequency")
    )

    #---- Monetary: total revenue (exluding returned items)---
    total_revenue = (
        df_net.groupby("cust_id")["sale_revenue"]
        .sum()
        .rename("total_revenue")
    )
    
    #---- Average ticket: revenue per effective order  ---
    avg_ticket = (total_revenue/frequency).rename("avg_ticket")

    #---- Items per order (gross): includes returned items, captures order behavior ---
    items_per_order_gross = (
        df.groupby(["cust_id","sale_id"])
        .size()
        .groupby("cust_id").mean()
        .rename("irems_per_order_gross")
    )

    #--- Items per order (net): excludes returned items, captures what they actually kept---
    items_per_order_net = (
        df_net.groupby(["cust_id","sale_id"])
        .size()
        .groupby("cust_id").mean()
        .rename("items_per_order_net")
    )

    
    #--- Return rate: absolute count and proportion proportion of items returned---
    total_returns = (
        df.groupby("cust_id")["is_returned"]
        .sum()
        .rename("total_returns")
    )

    total_items= (
        df.groupby("cust_id")["is_returned"]
        .count()
        .rename("total_items")
    )
    return_rate = (total_returns/total_items).rename("return_rate")

    #--- Average delivery time: days between order and packing ---
    df["delivery_days"]=(df["pack_date"]-df["order_date"]).dt.days
    avg_delivery = (
        df.groupby("cust_id")["delivery_days"]
        .mean()
        .rename("avg_delivery_days")
    )

    #--- Discount behaivor ---
    df["has_discount_item"] = (df["sale_discount_applied"]<0).astype(int)

    total_discounted_items = (
        df.groupby("cust_id")["has_discount_item"]
        .sum()
        .rename("total_discounted_items")
    )

    discount_rate= (
        df.groupby("cust_id")["has_discount_item"]
        .mean()
        .rename("discount_rate")
    )

    # Combine all features into one dataframe
    rfm = pd.concat([
        recency, frequency, total_revenue, avg_ticket, items_per_order_gross, 
        items_per_order_net, total_returns, total_items, return_rate, 
        total_discounted_items, discount_rate], axis=1).reset_index()
    return rfm

#Test it
rfm_train =compute_rfm_features(df_train)
print(rfm_train.shape)
rfm_train.head()
    
    

(93272, 12)


,cust_id,recency_days,frequency,total_revenue,avg_ticket,irems_per_order_gross,items_per_order_net,total_returns,total_items,return_rate,total_discounted_items,discount_rate
0,222agnowc53dykbq,383,1.0,89.95,89.950000,1.000,1.000000,0,1,0.000000,0,0.000000
1,222ny4m63rmalpdl,374,1.0,125.93,125.930000,3.000,2.000000,1,3,0.333333,3,1.000000
2,223xvc4rbjatlnev,520,1.0,116.14,116.140000,2.000,2.000000,0,2,0.000000,2,1.000000
3,223y2r357elerbis,348,1.0,47.97,47.970000,2.000,1.000000,1,2,0.500000,2,1.000000
4,2245y4r7mgo45qg3,169,7.0,536.70,76.671429,1.125,1.142857,1,9,0.111111,8,0.888889


## Seasonality features

In [19]:
def compute_seasonality_features(df):
    """
    Computes seasonality features, aggregated at customer level.
    Captures when during the year the customer tends to buy
    Input: transaction-level dataframe
    Output: one row per customer
    """
    df = df.copy()

    #Extract month from order date
    df["order_month"]= df["order_date"].dt.month

    #--- Does the customer buy in January or July? (peak sales months)---
    buys_in_jan = (
    df.groupby("cust_id")["order_month"]
    .apply(lambda x: int((x==1).any()))
    .rename("buys_in_jan")
    )

    buys_in_jul = (
    df.groupby("cust_id")["order_month"]
    .apply(lambda x:int((x==7).any()))
    .rename("buys_in_jul")
    )

    #--- Most active month: the month where the customer placed most orders---
    most_active_month = (
    df.groupby(["cust_id", "order_month"])["sale_id"]
    .nunique()
    .groupby("cust_id").idxmax()
    .apply(lambda x:x[1])
    .rename("most_active_month")
    )

    #Combine
    seasonality = pd.concat([buys_in_jan, buys_in_jul, most_active_month], axis=1).reset_index()
    return seasonality

#Test it
seasonality_train = compute_seasonality_features(df_train)
print(seasonality_train.shape)
seasonality_train.head()

(93272, 4)


,cust_id,buys_in_jan,buys_in_jul,most_active_month
0,222agnowc53dykbq,0,0,12
1,222ny4m63rmalpdl,0,0,12
2,223xvc4rbjatlnev,0,1,7
3,223y2r357elerbis,1,0,1
4,2245y4r7mgo45qg3,1,1,2
